# CDK9/CDK2-サイクリン複合体 (4BCF/4BCH/4BCI/4BCJ, 4BCK/4BCM/4BCN/4BCO/4BCQ) での chem.protein.split() デモ

同じ2-amino-4-heteroaryl-pyrimidine系阻害剤シリーズを、CDK9-サイクリンT複合体4構造
(`4BCF`/`4BCH`/`4BCI`/`4BCJ`)とCDK2-サイクリンA複合体5構造
(`4BCK`/`4BCM`/`4BCN`/`4BCO`/`4BCQ`)にそれぞれ結合させた計9構造を題材に、
`chem.protein.split()` の動作を確認する。ネプリライシンの単一チェーン構造より
題材として優れている点:

- **複数チェーンの複合体** -- CDK9セットはキナーゼ+サイクリンの2チェーン、CDK2セットは
  非対称単位に2コピー含む4チェーン構成で、デフォルト(チェーンごとに分割)の挙動を
  チェーン数が異なる構造にわたって確認できる
- **共有結合した修飾残基(`TPO`)** -- 活性化ループのリン酸化トレオニン(CDK9はThr186、
  CDK2はThr160)がHETATM `TPO` として記録されている。糖鎖修飾(`NAG`)とはまた別の
  「結合次数テンプレートと一致しない」ケース(遊離型アミノ酸テンプレートに対し、
  ペプチド結合で挟まれた主鎖内残基は原子組成自体は一致するが、後述のとおり実際には
  一貫してテンプレートマッチングに失敗する)として、`bond_orders_restored=False`の
  挙動を確認するのに使える
- **同一阻害剤シリーズが両ターゲットに共通** -- `T6Q`/`T7Z`/`T3E`/`T9N`の4化合物は
  CDK9・CDK2の両方に、`TJF`はCDK2のみに結合しており、標的選択性データのような
  比較ができる


In [ ]:
from chem import rcsb

entry_ids = ["4BCF", "4BCH", "4BCI", "4BCJ", "4BCK", "4BCM", "4BCN", "4BCO", "4BCQ"]
rcsb.download_structures(entry_ids, outdir="cdk9_cdk2_data", filetype="pdb")


## chem.protein.split() で全構造を分割する

`chem.protein.split()` は構造ファイルを (1) リガンドフリーの蛋白質PDB(デフォルトでは
チェーンごとに、ファイル名にチェーンIDを含めて分割。`all_chains=True`で1ファイルに
まとめることもできる)と (2) 水以外の各HETATM残基インスタンス全てのSDF(PDB Chemical
Component Dictionaryのテンプレートと照合できたものは結合次数・芳香族性を復元して
`bond_orders_restored=True`、できなかったものは警告付きで原子座標間の距離から推定した
単結合のみの生の結合情報を`bond_orders_restored=False`として — インスタンスが
スキップされることはない)、に分割する。9構造それぞれに適用する。ここでは
`remove_water=True` を指定し、結晶水(HETATM `HOH`)も蛋白質PDBから取り除く
(デフォルトの`remove_water=False`では結晶水は残る)。


In [ ]:
import os

from chem import protein

split_results = {}
for entry_id in entry_ids:
    split_results[entry_id] = protein.split(
        os.path.join("cdk9_cdk2_data", f"{entry_id}.pdb"),
        remove_water=True,
        outdir="cdk9_cdk2_split",
    )
split_results["4BCK"]


### チェーン構成の確認

CDK9セット(`4BCF`/`4BCH`/`4BCI`/`4BCJ`)はキナーゼ+サイクリンの2チェーン、CDK2セット
(`4BCK`/`4BCM`/`4BCN`/`4BCO`/`4BCQ`)は非対称単位に2コピーずつ含む4チェーンになっており、
`"protein"`(デフォルトの`{チェーンID: パス}`辞書)のキー数がそれを反映していることを確認する。


In [ ]:
import pandas as pd

chains_df = pd.DataFrame(
    [
        {"entry_id": entry_id, "n_chains": len(r["protein"]), "chain_ids": ", ".join(sorted(r["protein"]))}
        for entry_id, r in split_results.items()
    ]
)
chains_df


### 分割結果(リガンド)の一覧

各構造の`split()`結果(`"ligands"`)を1つの表にまとめる。`TPO`(リン酸化トレオニン、
主鎖内の修飾残基)は9構造・全インスタンスにわたって一貫して`bond_orders_restored=False`
(結合次数を復元できず生の結合情報で書き出し)になる一方、低分子阻害剤
(`T6Q`/`T7Z`/`T3E`/`T9N`/`TJF`)や結晶化添加剤(`GOL`/`SGM`/`SO4`)は`True`になることを確認する。


In [ ]:
rows = [
    {"entry_id": entry_id, **lig}
    for entry_id in entry_ids
    for lig in split_results[entry_id]["ligands"]
]
ligands_df = pd.DataFrame(rows)[
    ["entry_id", "code", "chain", "resnum", "bond_orders_restored", "path"]
]
ligands_df


In [ ]:
# コードごとの bond_orders_restored 集計 -- TPO だけが一貫して False になることを確認
ligands_df.groupby("code")["bond_orders_restored"].agg(["all", "count"]).rename(
    columns={"all": "always_restored", "count": "n_instances"}
)


### 網羅性確認: 水以外のHETATM残基が1つも欠けていないことを確認

`chem.ligand.list_ligand_instances` (`exclude=chem.protein.WATER`) で構造ファイル自身から
数えた水以外のHETATM残基インスタンス数と、`split()`が実際に書き出したSDF数が
一致することを9構造すべてで確認する(`TPO`のように結合次数を復元できないものも含め、
1つも欠けずSDF化されているはず)。


In [ ]:
from chem.ligand import list_ligand_instances
from chem.protein import WATER

for entry_id in entry_ids:
    all_instances = list_ligand_instances(os.path.join("cdk9_cdk2_data", f"{entry_id}.pdb"), exclude=WATER)
    n_written = len(split_results[entry_id]["ligands"])
    print(entry_id, f"{n_written}/{len(all_instances)} non-water HETATM instances written to SDF")
    assert n_written == len(all_instances)


### リガンドフリー蛋白質PDBの中身を確認

`"protein"`辞書の各チェーンファイルに対して改めて`list_ligand_instances`を実行し、
`TPO`を含め水以外のHETATM残基が本当に1つも残っていないことを確認する。


In [ ]:
n_checked = 0
for entry_id in entry_ids:
    for chain_id, protein_path in split_results[entry_id]["protein"].items():
        remaining = list_ligand_instances(protein_path, exclude=WATER)
        assert remaining == [], f"{entry_id} chain {chain_id}: {remaining}"
        n_checked += 1
print(f"checked {n_checked} protein PDB files -- no non-water HETATM residues remain in any of them")


### remove_water=True により結晶水も残っていないことを確認

上のセルは水以外のHETATM残基が残っていないことしか見ていない(`list_ligand_instances`は
そもそも水を対象外にしている)。`remove_water=True`を指定したので、蛋白質PDBには
`HOH`残基自体も1つも含まれていないはずであることを、`Bio.PDB`で直接数えて確認する。


In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
n_water_found = 0
for entry_id in entry_ids:
    for chain_id, protein_path in split_results[entry_id]["protein"].items():
        structure = parser.get_structure(entry_id, protein_path)
        n_water = sum(1 for res in next(structure.get_models()).get_residues() if res.id[0] == "W")
        n_water_found += n_water
        assert n_water == 0, f"{entry_id} chain {chain_id}: {n_water} water residue(s) remain"
print(f"checked {len(entry_ids)} structures -- {n_water_found} water residues remain across all protein PDBs")


### TPO(修飾残基, bond_orders_restored=False)と阻害剤(bond_orders_restored=True)を見比べる

`bond_orders_restored=False`のSDFは、RDKitが原子座標間の距離から推定した単結合のみの
生の結合情報である(芳香環があっても検出されない)のに対し、`True`のSDFは
PDB Chemical Component Dictionaryのテンプレートで結合次数・芳香族性が正しく
復元されていることを、実際のSDFを読み込んで比較する。


In [ ]:
from rdkit import Chem

tpo_entry = next(lig for lig in split_results["4BCF"]["ligands"] if lig["code"] == "TPO")
t6q_entry = next(lig for lig in split_results["4BCF"]["ligands"] if lig["code"] == "T6Q")

for label, entry in [("TPO (bond_orders_restored=False)", tpo_entry), ("T6Q (bond_orders_restored=True)", t6q_entry)]:
    mol = next(Chem.SDMolSupplier(entry["path"], sanitize=entry["bond_orders_restored"]))
    bond_types = {str(b.GetBondType()) for b in mol.GetBonds()}
    n_aromatic = sum(atom.GetIsAromatic() for atom in mol.GetAtoms())
    print(f"{label}: {mol.GetNumAtoms()} atoms, bond types={bond_types}, {n_aromatic} aromatic atoms")


### 阻害剤シリーズをターゲット横断で比較する(分子量・QED)

`T6Q`/`T7Z`/`T3E`/`T9N`はCDK9・CDK2の両方に、`TJF`はCDK2のみに結合している。
同じ化合物コードは同じ分子なので、コードごとに1エントリだけ読み込み直し、
どちらのターゲットに結合していたかと合わせて比較する。


In [ ]:
from chem import ligand

main_ligand_codes = {"T6Q", "T7Z", "T3E", "T9N", "TJF"}

rows = []
seen_codes = set()
for entry_id in entry_ids:
    for lig in split_results[entry_id]["ligands"]:
        if lig["code"] not in main_ligand_codes or lig["code"] in seen_codes:
            continue
        seen_codes.add(lig["code"])
        mol = next(Chem.SDMolSupplier(lig["path"]))
        bound_to = sorted(
            {e for e in entry_ids for l2 in split_results[e]["ligands"] if l2["code"] == lig["code"]}
        )
        rows.append(
            {
                "code": lig["code"],
                "bound_to_entries": ", ".join(bound_to),
                "n_atoms": mol.GetNumAtoms(),
                "molecular_weight": round(ligand.molecular_weight(mol), 1),
                "qed": round(ligand.qed(mol), 3),
                "smiles": Chem.MolToSmiles(mol),
            }
        )

pd.DataFrame(rows).sort_values("code").reset_index(drop=True)


### 可視化: 4チェーン複合体(4BCK)のリガンドフリー蛋白質 + 2つの阻害剤SDFを重ねて表示

`4BCK`はCDK2-サイクリンA複合体の非対称単位に2コピー(キナーゼ: チェーンA/C、
サイクリン: チェーンB/D)を含む。4本の蛋白質チェーン(色分け)と、両キナーゼコピーに
結合した阻害剤`T3E`のSDF(stick)を同じpy3Dmolビューに重ねて表示する。


In [ ]:
import py3Dmol

chain_colors = {"A": "lightblue", "B": "wheat", "C": "lightgreen", "D": "lightpink"}

view = py3Dmol.view(width=650, height=500)
n_models = 0
for chain_id, protein_path in sorted(split_results["4BCK"]["protein"].items()):
    with open(protein_path) as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": n_models}, {"cartoon": {"color": chain_colors[chain_id]}})
    n_models += 1

ligand_model_ids = []
for lig in split_results["4BCK"]["ligands"]:
    if lig["code"] != "T3E":
        continue
    with open(lig["path"]) as f:
        view.addModel(f.read(), "sdf")
    view.setStyle({"model": n_models}, {"stick": {"colorscheme": "orangeCarbon"}})
    ligand_model_ids.append(n_models)
    n_models += 1

view.zoomTo({"model": ligand_model_ids})
view.show()


## chem.protein.identity_matrix() で分割済みチェーン同士の配列一致度を確認する

`chem.protein.split()`が書き出した蛋白質PDBは、9構造・計28ファイル(チェーンごとに1
ファイル)ある。`chem.protein.identity_matrix()`はこれらを総当たりで配列比較し、
`{path_i: {path_j: identity, ...}, ...}`という対称なマトリクスを返す(`chem.protein.align`
と同じ`identity`の定義だが、reference/構造的重ね合わせは行わない)。同一蛋白質の複数コピー
(例: 4構造それぞれのCDK9キナーゼチェーン、あるいは`4BCK`の非対称単位内の2コピーのCDK2キナーゼ)
は~1.0になる一方、キナーゼ vs サイクリンは全く無関係な蛋白質(異なるフォールド)なので
~0.15程度と低く出る。ただしCDK9とCDK2のキナーゼドメイン同士は、同じCDK(サイクリン依存性
キナーゼ)ファミリーに属するパラログ同士のため、キナーゼ vs サイクリンほど低くはならず
~0.4前後の中程度の一致度になる -- 「無関係」と「遠縁のパラログ」の違いがマトリクス上に
現れることも合わせて確認する。ヒートマップにすると、CDK9キナーゼ・サイクリンT・
CDK2キナーゼ・サイクリンAという4つの高一致度ブロックが対角線上にはっきり見えるはずである。


In [ ]:
import numpy as np

labels = []
protein_paths = []
for entry_id in entry_ids:
    for chain_id, path in sorted(split_results[entry_id]["protein"].items()):
        labels.append(f"{entry_id}_{chain_id}")
        protein_paths.append(path)

identity = protein.identity_matrix(protein_paths)
identity_matrix_arr = np.array([[identity[p1][p2] for p2 in protein_paths] for p1 in protein_paths])
identity_matrix_arr.shape


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(identity_matrix_arr, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
fig.colorbar(im, ax=ax, label="sequence identity", fraction=0.046, pad=0.04)
ax.set_title("chem.protein.identity_matrix() over all 28 split protein chains")
plt.tight_layout()
plt.show()
